# 1. Initializations

## 1.1 General CPU/GPU Checks (NVIDIA cards)

In [ ]:
### global
import logging
import os
import shutil
from datetime import datetime
from pathlib import Path
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
print(f'Path [{os.environ["PATH"]}]')

# Test pytorch GPU config
import torch
cuda_test = torch.cuda.is_available()
print(f"✅ Torch CUDA available: {cuda_test}")
device_name_gpu = torch.cuda.get_device_name(0)
device_gpu = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
print(f"🖥️ Device Name: {device_name_gpu} | Device reference: {device_gpu}")

## 1.2 General imports

In [ ]:
# Pour la manipulation de tableaux et Dataframes
from PIL import Image
import numpy as np

# modelisation
from scipy.ndimage import zoom
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    Trainer, TrainingArguments, EvalPrediction,  # type: ignore
    AutoConfig,
)
import torch
from torch.utils.data import Dataset, random_split
from torchvision import datasets
import evaluate
from captum.attr import LayerIntegratedGradients


# Pour la visualisation des performances
import matplotlib.pyplot as plt
%matplotlib inline

# 2. Loading and Data Enrichment

In [ ]:
class HFDatasetWrapper(Dataset):
    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        # image est un PIL.Image car pas de transform appliqué
        inputs = self.processor(images=image, return_tensors="pt", do_rescale=False)
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["label"] = label
        return inputs


class CachedDataset(Dataset):
    def __init__(self, dataset):
        print("📥 Mise en cache en mémoire...")
        self.samples = [dataset[i] for i in range(len(dataset))]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
def classer_images_par_age(
    repertoire_images,
    age_min=None,
    age_max=None
):
    """
    Organise les images dans des sous-dossiers en fonction de l'âge extrait du nom de fichier.
    - Peut filtrer une plage d'âges : age_min à age_max
    - Réindexe les âges sélectionnés en commençant à 0
    """

    ages_valides = set()

    # Étape 1 — Parcourir tous les fichiers et collecter les âges valides
    fichiers_eligibles = []
    for racine, _, fichiers in os.walk(repertoire_images):
        for fichier in fichiers:
            if fichier.lower().endswith(".jpg"):
                try:
                    age = int(fichier.split("_")[0])
                    if ((age_min is None or age >= age_min) and
                        (age_max is None or age <= age_max)):
                        fichiers_eligibles.append((fichier, age, racine))
                        ages_valides.add(age)
                except Exception as e:
                    print(f"⚠️ Ignoré : {fichier} (erreur : {e})")

    # Étape 2 — Réindexer les âges valides
    ages_valides = sorted(ages_valides)
    mapping_ages = {age: idx for idx, age in enumerate(ages_valides)}
    print(f"🎯 Mapping des âges : {mapping_ages}")

    # Étape 3 — Déplacer les fichiers vers les bons dossiers (réindexés)
    for fichier, age, racine in fichiers_eligibles:
        nouvelle_classe = str(mapping_ages[age])
        dest_dir = os.path.join(repertoire_images, nouvelle_classe)
        os.makedirs(dest_dir, exist_ok=True)

        chemin_source = os.path.join(racine, fichier)
        chemin_destination = os.path.join(dest_dir, fichier)

        try:
            shutil.move(chemin_source, chemin_destination)
        except Exception as e:
            print(f"❌ Erreur déplacement {fichier} : {e}")

    print("✅ Organisation terminée.")

In [ ]:
# dataset provenant de https://susanqq.github.io/UTKFace/ et décompressé localement avec application d'une reconstruction
# des sous-répertoires par classe avec la fonction utilitaire classer_images_par_age dans le but d'être compatible 
# avec image_dataset_from_directory
# Ajuster le nom du répertoire ou sont décompressés les images de visage et selectionner la plage pour les Ages en fonction
# des performance de votre machine (ajusté pour GPU RTX3080 ici)
data_dir = "C:\\Users\\remyc\\Downloads\\visages\\"  
min_age = 25
max_age = 45
classer_images_par_age(data_dir, min_age, max_age)

# 3. Deep learning

In [ ]:
def load_vision_dataset_for_trainer(
    data_dir: str,
    pretrained_model_name_or_path: str,
    cache_in_memory: bool = True,
    split_ratio: float = 0.8,
    seed: int = 42,
):
    processor = AutoImageProcessor.from_pretrained(pretrained_model_name_or_path, use_fast=True)
    
    base_dataset = datasets.ImageFolder(root=data_dir)

    label2id = base_dataset.class_to_idx
    id2label = {idx: f"{int(label)+min_age} ans" for label, idx in label2id.items()}

    num_classes = len(label2id)
    print(f"🧩 Détection automatique : {num_classes}\n"
          f"label2id → {label2id}\n"
          f"id2label → {id2label}\n")

    train_len = int(split_ratio * len(base_dataset))
    test_len = len(base_dataset) - train_len
    train_raw, test_raw = random_split(
        base_dataset, 
        [train_len, test_len],
        generator=torch.Generator().manual_seed(seed)
    )

    train_ds = HFDatasetWrapper(train_raw, processor)
    test_ds = HFDatasetWrapper(test_raw, processor)

    if cache_in_memory:
        train_ds = CachedDataset(train_ds)
        test_ds = CachedDataset(test_ds)

    return train_ds, test_ds, processor, num_classes, id2label, label2id

## 3.1 Modèle basé sur Layer PyTorch (transfert learning)

#### Creation & Execution

In [ ]:
# === Chargement du modèle et du processeur ===
MODEL_PATH = "nn_pytorch_hf.model"
TL_MODEL_NAME = "nateraw/vit-age-classifier"
if os.path.exists(MODEL_PATH):
    print(f"🔁 Chargement du modèle depuis {MODEL_PATH} pour continuer l'entrainement")
    train_dataset, test_dataset, model_processor, num_classes, id2label, label2id = load_vision_dataset_for_trainer(
        data_dir=data_dir,
        pretrained_model_name_or_path=MODEL_PATH,
        cache_in_memory=False
    )
    config = AutoConfig.from_pretrained(MODEL_PATH)
    config.num_labels = num_classes
    config.id2label = id2label
    config.label2id = label2id
    nn_pytorch_hf = AutoModelForImageClassification.from_pretrained(
        MODEL_PATH, 
        config=config,
        ignore_mismatched_sizes=True
    )
else:
    train_dataset, test_dataset, model_processor, num_classes, id2label, label2id = load_vision_dataset_for_trainer(
        data_dir=data_dir,
        pretrained_model_name_or_path=TL_MODEL_NAME,
        cache_in_memory=False
    )
    config = AutoConfig.from_pretrained(TL_MODEL_NAME)
    config.num_labels = num_classes
    config.id2label = id2label
    config.label2id = label2id
    nn_pytorch_hf = AutoModelForImageClassification.from_pretrained(
        TL_MODEL_NAME, 
        config=config,
        ignore_mismatched_sizes=True
    )

In [ ]:
print(nn_pytorch_hf)

#### Entraînement et métriques

In [ ]:
model_metric = evaluate.load("accuracy")
def compute_metrics_method(eval_pred: EvalPrediction) -> dict:
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return model_metric.compute(predictions=predictions, references=labels)  # type: ignore

In [ ]:
new_run_dir = f"runs.model/nn_pytorch_hf_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
model_training_args = TrainingArguments(
    output_dir=new_run_dir,              # chemin de stockage des poids du modèle
    num_train_epochs=1,              # nombre d'époques pour l'entraînement
    per_device_train_batch_size=16,  # batch size pour l'entraînement
    per_device_eval_batch_size=32,   # batch size pour l'évaluation du modèle
    learning_rate=1e-4,              # taux d'apprentissage
    weight_decay=0.01,               # paramètre décidant des poids
    logging_dir=new_run_dir,             # chemin de stockage des logs
    # utilisation du meilleur modèle à l'issue de l'entraînement
    load_best_model_at_end=True,     # utilisation du meilleur modèle à l'issue de l'entraînement
    logging_steps=300,               # log & enregistrer les poids à chaque 400 itérations
    save_strategy="epoch",           # sauvegarde à chaque epoch
    eval_strategy="epoch"            # évaluation à chaque epoch
)

In [ ]:
# resume_path = "runs.model/nn_pytorch_hf_20250708-170000/checkpoint-1122"
resume_path=""
if resume_path:
    print(f"🔁 Reprise depuis le checkpoint : {resume_path}")
    model = AutoModelForImageClassification.from_pretrained(resume_path)
else:
    print(f"🚀 Nouvelle initialisation depuis {TL_MODEL_NAME}")
    model = nn_pytorch_hf

model_trainer = Trainer(
    model=model,
    args=model_training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics_method,
)
model_trainer.train(resume_from_checkpoint=resume_path if os.path.exists(resume_path) else None)

#### Sauvegarde du modèle

In [ ]:
nn_pytorch_hf.save_pretrained(MODEL_PATH)
model_processor.save_pretrained(MODEL_PATH)

#### Prédiction et évaluation

In [ ]:
# === Chargement et prétraitement des images ===
img_paths = [
    Path("C:/Users/remyc/Downloads/visage_perso/25_remy.jpg"),
    Path("C:/Users/remyc/Downloads/visage_perso/44_remy.jpg")
]
images = [Image.open(p).convert("RGB") for p in img_paths]
inputs = model_processor(images, return_tensors="pt")

# === Device management ===
nn_pytorch_hf.to(device_gpu)
inputs = {k: v.to(device_gpu) for k, v in inputs.items()}

# === Prédiction ===
with torch.no_grad():
    outputs = nn_pytorch_hf(**inputs)
    outputs_probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

# === Affichage des probabilités ===
for idx, img_probs in enumerate(outputs_probs):
    print(f"\n📷 Image : {img_paths[idx].name}")
    
    # Récupérer les classes triées par proba décroissante
    sorted_indices = torch.argsort(img_probs, descending=True)
    
    for i in sorted_indices:
        class_id = int(i.item())
        label = nn_pytorch_hf.config.id2label.get(class_id, f"Label {class_id}")
        confidence = img_probs[class_id].item()
        print(f"  {label:<20} : {confidence:.2%}")

# 4. Interprétabilité

In [ ]:
def deprocess_tensor_for_display(tensor_img, mean, std):
    """Dénormalise un tenseur d'image [3, H, W] et le convertit pour affichage"""
    if tensor_img.ndim != 3 or tensor_img.shape[0] != 3:
        raise ValueError("L'image doit être un tensor [3, H, W]")

    # Convertir mean/std en tensors et broadcast
    mean = torch.tensor(mean).view(-1, 1, 1)
    std = torch.tensor(std).view(-1, 1, 1)

    # Dénormaliser
    img = tensor_img * std + mean
    img = img.clamp(0, 1)  # pour éviter les dépassements

    # Convertir en format [H, W, C] pour affichage
    img = img.permute(1, 2, 0).cpu().numpy()
    return img

In [ ]:
def show_importance(model, interpretor, input_tensor, target=0, device="cuda", **attribute_kwargs):
    input_tensor = input_tensor.to(device).unsqueeze(0)  # [1, C, H, W]
    model.to(device)
    model.eval()

    attributions = interpretor.attribute(inputs=input_tensor, target=target, **attribute_kwargs)
    heatmap = attributions.sum(dim=1).squeeze(0)  # [H, W]

    heatmap = heatmap.cpu().detach().numpy()
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) != 0:
        heatmap /= np.max(heatmap)

    return heatmap  # [H, W] float in [0, 1]

In [ ]:
def apply_importance(img: np.ndarray, heatmap: np.ndarray, alpha: float = 0.4, resize: bool = True) -> np.ndarray:
    """
    Superpose une heatmap (H, W) sur une image (H, W, 3) avec fusion alpha.

    Args:
        img (np.ndarray): Image RGB de forme (H, W, 3), valeurs entre 0 et 1.
        heatmap (np.ndarray): Carte d'importance de forme (h, w) ou (H, W).
        alpha (float): Poids de la heatmap dans la fusion.
        resize (bool): Si True, redimensionne la heatmap à la taille de l'image.

    Returns:
        np.ndarray: Image fusionnée RGB de forme (H, W, 3).
    """
    if heatmap.ndim != 2:
        raise ValueError("La heatmap doit être une image 2D [H, W]")

    if resize:
        zoom_factors = (
            img.shape[0] / heatmap.shape[0],
            img.shape[1] / heatmap.shape[1],
        )
        heatmap = zoom(heatmap, zoom_factors, order=1)  # bilinear resize

    # Normalisation défensive
    heatmap = np.clip(heatmap, 0, 1)

    # Appliquer la heatmap sur l'image (colormap rouge par défaut)
    heatmap_rgb = np.zeros_like(img)
    heatmap_rgb[..., 0] = heatmap  # canal rouge

    # Fusion alpha
    overlay = (1 - alpha) * img + alpha * heatmap_rgb
    overlay = np.clip(overlay, 0, 1)
    
    return overlay

In [ ]:
def forward_logits_only(inputs):
    outputs = nn_pytorch_hf(inputs)
    return outputs.logits

# Placer le modèle et la target_layer sur CPU
nn_pytorch_hf = nn_pytorch_hf.to(device_cpu)
target_layer = nn_pytorch_hf.vit.embeddings.patch_embeddings.projection # 1ère couche convolutive

lig = LayerIntegratedGradients(forward_func=forward_logits_only, layer=target_layer)

# 📊 Affichage côte à côte
nb_images = inputs["pixel_values"].shape[0]
fig, axes = plt.subplots(nrows=nb_images, ncols=3, figsize=(12, 4 * nb_images))

# Toujours avoir une liste 2D de sous-axes : [[ax1, ax2], [ax3, ax4], ...]
if nb_images == 1:
    axes = [axes]  # axes est (ax1, ax2), on le transforme en [ (ax1, ax2) ]
# axes = [list(row) if not isinstance(row, list) else row for row in axes]  # en liste explicite

for i, ax_row in enumerate(axes):
    input_tensor = inputs["pixel_values"][i].to(device_cpu)

    # Déterminer la target automatiquement
    predicted_class = int(torch.argmax(outputs_probs[i]).item())
    predicted_label = nn_pytorch_hf.config.id2label[predicted_class]

    print(f"🔎 Image {i} — classe prédite : {predicted_label} ({outputs_probs[i][predicted_class]:.2%})")

    # calcul de la heatmap pour le layer donné
    heatmap = show_importance(nn_pytorch_hf, lig, 
                              input_tensor, target=predicted_class, 
                              device=str(device_cpu),
                              return_convergence_delta=False, 
                              n_steps=25)
          
    # Image dénormalisée
    mean = model_processor.image_mean
    std = model_processor.image_std
    img_display = deprocess_tensor_for_display(input_tensor, mean, std)

    # image superposée à la heatmap
    img_overlayed = apply_importance(img_display, heatmap, alpha=0.4)

    # 📷 Image originale
    ax_row[0].imshow(img_display)
    ax_row[0].set_title(f"Image d'origine {i}")
    ax_row[0].axis("off")

    # 🔥 Heatmap seule (pas superposée ici)
    ax_row[1].imshow(heatmap, cmap="jet")
    ax_row[1].set_title(f"Grad-CAM (target={predicted_label}) {i}")
    ax_row[1].axis("off")

    # 🔥 Heatmap superposée à l'image
    ax_row[2].imshow(img_overlayed, cmap="jet")
    ax_row[2].set_title(f"Grad-CAM (target={predicted_label}) {i}")
    ax_row[2].axis("off")

plt.tight_layout()
plt.show()